# Unzip file

In [1]:
# import zipfile

# # path to the zip file
# zip_path = r'D:\Ds108\Lab4\Data for practice-20260417T070843Z-3-001.zip'
# # directory where you want to extract the files
# extract_path = r'D:\Ds108\Lab4\raw_data'

# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)


# Preprocess

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Pandas version:", pd.__version__)

Pandas version: 2.2.2


In [3]:
# Khai báo đường dẫn gốc
BASE_PATH = r'D:\Ds108\Lab4\raw_data\Data for practice'

# Load tập Tháng 4-6 (Sẽ dùng làm Train set)
df_delay46 = pd.read_csv(f'{BASE_PATH}\delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv')
df_notdelay46 = pd.read_csv(f'{BASE_PATH}\\not_delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv')

# Load tập Tháng 7-9 (Sẽ dùng làm Test set / Out-of-Time Validation)
df_delay79 = pd.read_csv(f'{BASE_PATH}\delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv')
df_notdelay79 = pd.read_csv(f'{BASE_PATH}\\not_delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv')

# Gộp dữ liệu theo giai đoạn
df_train_raw = pd.concat([df_delay46, df_notdelay46], ignore_index=True)
df_test_raw = pd.concat([df_delay79, df_notdelay79], ignore_index=True)

# Lấy các cột chung để đảm bảo Train/Test đồng nhất
common_cols = df_train_raw.columns.intersection(df_test_raw.columns)
df_train_raw = df_train_raw[common_cols].copy()
df_test_raw = df_test_raw[common_cols].copy()

print(f"Shape of Train (Apr-Jun): {df_train_raw.shape}")
print(f"Shape of Test (Jul-Sep): {df_test_raw.shape}")

Shape of Train (Apr-Jun): (399053, 37)
Shape of Test (Jul-Sep): (1074897, 37)


In [ ]:
def normalize_columns(df):
    """Làm sạch tên cột: Xóa khoảng trắng thừa, thay dấu cách bằng underscore"""
    df = df.copy()
    cols = (
        pd.Series(df.columns)
        .astype(str)
        .str.strip()
        .str.replace(r'\s+', '_', regex=True)
    )
    df.columns = cols
    # Xóa các cột bị duplicate (nếu có sau khi đổi tên)
    df = df.loc[:, ~df.columns.duplicated()]
    return df

def drop_unnecessary_columns(df):
    df = df.copy()
    
    # 1. DROP Data Leakage 
    leakage_cols = ['REASON_CD', 'SOUF_RCV_NO', 'QTUF_RCV_NO', 'SHIP_DECISION_NO']
    
    # 2. DROP Useless (Thêm Sales_order_line_number vào đây)
    useless_cols = ['SUBSIDIARY_CD', 'INNER_CD', 'GLOBAL_NO', 'Sales_order_line_number']
    
    cols_to_drop = leakage_cols + useless_cols
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
    
    return df

In [5]:
def clean_and_cast_types(df):
    """Xử lý missing value và ép kiểu dữ liệu chuẩn bị cho EDA/Modeling"""
    df = df.copy()
    
    # 1. Xử lý Missing Value cho Categorical
    if 'Ship_Mode' in df.columns:
        df['Ship_Mode'] = df['Ship_Mode'].fillna('Missing')
        
    if 'SUPPLIER_DIV' in df.columns:
        df['SUPPLIER_DIV'] = df['SUPPLIER_DIV'].fillna(-1)
        
    # 2. Xử lý các cột cờ (Flags) bị dính khoảng trắng/Null
    if 'OTHER_AREA_SHIP_DIV' in df.columns:
        df['OTHER_AREA_SHIP_DIV'] = pd.to_numeric(
            df['OTHER_AREA_SHIP_DIV'].replace(' ', np.nan), errors='coerce'
        ).fillna(0)
        
    if 'Consider_count_hodiday_Saturday' in df.columns: # Giữ nguyên tên gốc bị sai chính tả của data
        df['Consider_count_hodiday_Saturday'] = pd.to_numeric(
            df['Consider_count_hodiday_Saturday'], errors='coerce'
        ).fillna(0)
        
    # 3. Ép kiểu Datetime cho Order_date và VSD (Rất quan trọng cho Notebook 2)
    df['Order_date'] = pd.to_datetime(df['Order_date'], errors='coerce')
    df['VSD'] = pd.to_datetime(df['VSD'], errors='coerce')
    
    # 4. ÉP KIỂU CATEGORY CHO LIGHTGBM (Thần chú tăng điểm)
    # Gom cả HIGH_CARD và LOW_CARD lại, để LightGBM tự dùng thuật toán Fisher xử lý
    categorical_cols = ['PRODUCT_CD', 'BRAND_CD', 'SUPPLIER_CD', 'Ship_Mode', 'DELI_DIV', 'PACKING_RANK', 'Stock_class']
    
    for col in categorical_cols:
        if col in df.columns:
            # Chuyển về string trước để đồng nhất, sau đó ép sang category
            df[col] = df[col].astype(str).astype('category')
            
    return df

In [6]:
def run_preprocessing_pipeline(df):
    df = normalize_columns(df)
    df = drop_unnecessary_columns(df)
    df = clean_and_cast_types(df)
    return df

print("Processing Train Set...")
train_clean = run_preprocessing_pipeline(df_train_raw)

print("Processing Test Set...")
test_clean = run_preprocessing_pipeline(df_test_raw)

# Kiểm tra lại Missing values sau khi clean
print("\nMissing values in Train set (Top 5):")
print(train_clean.isna().sum().sort_values(ascending=False).head(5))

Processing Train Set...
Processing Test Set...

Missing values in Train set (Top 5):
Order_date         0
SPECIAL_DIV        0
SO_DAY_OF_WEEK     0
SO_DAY_OF_MONTH    0
SUPPLIER_DIV       0
dtype: int64


In [7]:
# Tạo thư mục processed_data nếu chưa có
import os
processed_dir = r'D:\Ds108\Lab4\processed_data'
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)

# Save ra Parquet để giữ nguyên định dạng kiểu 'category' và 'datetime' (Nhanh và nhẹ hơn CSV rất nhiều)
train_clean.to_parquet(f'{processed_dir}\\train_cleaned.parquet', index=False)
test_clean.to_parquet(f'{processed_dir}\\test_cleaned.parquet', index=False)

print(f"Saved successfully to {processed_dir}")

Saved successfully to D:\Ds108\Lab4\processed_data
